# Exercise 5 (Task 1)

### Minimize the Ising Hamiltonian with a Genetic Algorithm (GA)

Use a **Genetic Algorithm** to find a low-energy spin configuration for the 2D Ising model.

**Model (periodic boundary conditions):**  
$H(\mathbf{s}) = -J \sum_{\langle i,j\rangle} s_i s_j \;-\; h \sum_i s_i,\qquad s_i \in \{-1,+1\},$
where $\langle i,j\rangle$ denotes nearest neighbors on an $N \times N$ grid.

**Encoding:** a binary genome of length $N^2$. Each bit $b_i \in \{0,1\}$ is mapped to a physical spin  
$s_i = 2 b_i - 1 \in \{-1,+1\}.$

**Objective:** minimize $H(\mathbf{s})$.

Implement GA operators (selection, crossover, mutation), evolve the population, and report the best energy and corresponding spin configuration.


In [ ]:
import numpy as np, random, matplotlib.pyplot as plt

# ---------------- Problem setup ----------------
N = 16                 # lattice size (N x N)
J = 1.0                # coupling (>0 ferromagnetic)
h = 0.0                # external field
L = N * N              # genome length (bits)

random.seed(123); np.random.seed(123)

def bits_to_spins(bits):
    """Map {0,1} -> {-1,+1}."""
    return 2*bits - 1

def energy(bits):
    """Ising Hamiltonian with periodic boundary conditions."""
    s = bits_to_spins(bits).reshape(N, N)
    # nearest neighbors (right, down) with wrap-around (periodic BCs)
    s_right = np.roll(s, shift=-1, axis=1)
    s_down  = np.roll(s, shift=-1, axis=0)
    # H = -J sum_{<i,j>} s_i s_j - h sum_i s_i
    return float(-J*np.sum(s*s_right + s*s_down) - h*np.sum(s))

# ---------------- GA configuration ----------------
POP   = 60
GENS  = 200
TOURN = 3
PC    = 0.9            # crossover rate
PM    = 1.0 / L        # per-bit mutation rate (~1 flip per child on avg)
ELIT  = 2

def rand_ind():
    """Random individual: bitstring of length L."""
    return np.random.randint(0, 2, size=L, dtype=np.int8)

### TODO1: Fitness

In [ ]:
# ---------------- TODO 1: fitness (minimize energy) ----------------
def fitness(ind):
    """
    Return a scalar to MINIMIZE: the Ising energy of 'ind'.
    """
    # TODO: implement and return the energy of 'ind'
    raise NotImplementedError


### TODO2: Tournament selection

In [ ]:
# ---------------- TODO 2: tournament selection ----------------
def tournament_select(pop, losses, k=TOURN):
    """
    Pick k random indices; return a COPY of the individual with smallest loss.
    """
    # TODO: implement tournament selection and return a copy of the winner
    raise NotImplementedError


### TODO3:

In [ ]:
# ---------------- TODO 3: crossover (uniform) ----------------
def crossover(p1, p2):
    """
    With probability PC:
      - Create a random 0/1 mask M of length L (Bernoulli(0.5)).
      - c1 = where(M, p1, p2); c2 = where(M, p2, p1)
    Otherwise: return copies of parents.
    """
    # TODO: implement uniform crossover
    raise NotImplementedError


### TODO 4: Mutation

In [ ]:
# ---------------- TODO 4: mutation (bit-flip) ----------------
def mutate(ind):
    """
    Flip each bit with probability PM. Return a NEW individual.
    """
    # TODO: implement bit-flip mutation
    raise NotImplementedError

###TODO5: GA Loop

In [ ]:
# ---------------- GA loop ----------------
population = [rand_ind() for _ in range(POP)]
best_hist = []

for gen in range(GENS):
    # Evaluate population
    losses = [fitness(ind) for ind in population]
    order = np.argsort(losses)              # ascending (lower = better)
    population = [population[i] for i in order]
    losses    = [losses[i] for i in order]
    best_hist.append(losses[0])

    if gen % 20 == 0:
        print(f"Gen {gen:03d} | best H = {losses[0]:.1f}")

    # TODO 5: build next generation with elitism + offspring
    # Steps:
    #   1) Start new_pop with ELIT best individuals (copy them).
    #   2) While len(new_pop) < POP:
    #        - Select p1, p2 via tournament_select(...)
    #        - Perform crossover(p1, p2) -> c1, c2
    #        - Mutate c1 (and c2 if space remains)
    #        - Append to new_pop
    #   3) Replace population with new_pop
    raise NotImplementedError

# ---------------- Report & visualize (runs after TODOs are done) ----------------
best = population[0]
Hbest = fitness(best)
print(f"\nBest energy: {Hbest:.1f}")

spin = bits_to_spins(best).reshape(N,N)
plt.figure(figsize=(4.2,4.2))
plt.imshow(spin, cmap='bwr', vmin=-1, vmax=1)
plt.colorbar(label='spin')
plt.title(f"Best spin configuration (H={Hbest:.1f})")
plt.axis('off'); plt.tight_layout(); plt.show()

plt.figure(figsize=(5.2,3))
plt.plot(best_hist, lw=2)
plt.xlabel("Generation"); plt.ylabel("Best energy")
plt.title("GA Convergence — Ising model")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Exercise 5 (Task2)


### Rediscovering a Damped Oscillator from Data
You will use **Genetic Programming (GP)** to rediscover a compact expression that fits noisy measurements of a damped oscillator.

**Target family (unknown to GP):**  
$$x(t) \approx e^{-a t}\,\cos(bt + c).$$

**Dataset:** `data.csv` (200 samples, columns: `t, x`).

Fill every `# TODO` cell. Keep randomness initialized for reproducibility.

In [ ]:
## Upload the data file manually to your colab if you want to use colab instead of your local computer
from google.colab import files
uploaded = files.upload()

### Load the data

In [ ]:
# ============================================================
#  GP Symbolic Regression (Damped Oscillator)
# ============================================================

import math, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(123)
np.random.seed(123)

# -----------------------------
# Load or generate data
# -----------------------------
def load_or_make_data():
    fname = "data.csv"   # upload manually to colab or place in working dir
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        t = df["t"].values.astype(float)
        x = df["x"].values.astype(float)
        print(f"Loaded data from {fname}:", df.shape)
        return t, x
    # Synthetic damped cosine data
    A, gamma, omega, phi = 1.0, 0.15, 2.0, 0.3
    t = np.linspace(0, 10, 200)
    x_clean = A * np.exp(-gamma * t) * np.cos(omega * t + phi)
    x = x_clean + np.random.normal(0, 0.02, size=len(t))
    print("Generated synthetic data (200 pts).")
    return t, x

t, y = load_or_make_data()

### protected ops and set

In [ ]:
# -----------------------------
# Protected ops and primitive set
# -----------------------------
def p_div(a, b): return a / b if abs(b) > 1e-12 else a
def p_exp(a): return math.exp(max(-20.0, min(20.0, a)))
def p_sin(a): return math.sin(a)
def p_cos(a): return math.cos(a)

FUN_SET = {
    '+': (2, lambda a,b: a+b, '+'),
    '-': (2, lambda a,b: a-b, '-'),
    '*': (2, lambda a,b: a*b, '*'),
    '/': (2, p_div, '/'),
    'sin': (1, p_sin, 'sin'),
    'cos': (1, p_cos, 'cos'),
    'exp': (1, p_exp, 'exp'),
}
TERMINALS = ['t', 1.0, 2.0, 0.5, 0.1]

# -----------------------------
# Tree utilities
# -----------------------------
def is_terminal(node): return (node == 't') or isinstance(node, (int, float))

def tree_size(node):
    if is_terminal(node): return 1
    ar = FUN_SET[node[0]][0]
    if ar == 1: return 1 + tree_size(node[1])
    return 1 + tree_size(node[1]) + tree_size(node[2])

def tree_depth(node):
    if is_terminal(node): return 1
    ar = FUN_SET[node[0]][0]
    if ar == 1: return 1 + tree_depth(node[1])
    return 1 + max(tree_depth(node[1]), tree_depth(node[2]))

def pretty(node):
    if node == 't': return 't'
    if isinstance(node, (int, float)): return f"{float(node):.3g}"
    op = node[0]; ar, _, sym = FUN_SET[op]
    if ar == 1: return f"{sym}({pretty(node[1])})"
    return f"({pretty(node[1])} {sym} {pretty(node[2])})"

# -----------------------------
# Random tree generation
# -----------------------------
MAX_DEPTH = 7  # enough to express exp(-a t)*cos(b t + c)

def rand_terminal():
    # 70% palette constants/variable, 30% ephemeral numeric constant in [-5,5]
    if random.random() < 0.7:
        return random.choice(TERMINALS)
    return float(np.random.uniform(-5.0, 5.0))

def rand_function():
    return random.choice(list(FUN_SET.keys()))

def grow(depth=1):
    if depth >= MAX_DEPTH or (depth > 1 and random.random() < 0.3):
        return rand_terminal()
    op = rand_function()
    ar = FUN_SET[op][0]
    if ar == 1: return (op, grow(depth+1))
    return (op, grow(depth+1), grow(depth+1))

# -----------------------------
# Evaluate tree
# -----------------------------
def eval_tree(node, t_val):
    if node == 't': return float(t_val)
    if isinstance(node, (int, float)): return float(node)
    op = node[0]; ar, fn, _ = FUN_SET[op]
    if ar == 1: return fn(eval_tree(node[1], t_val))
    return fn(eval_tree(node[1], t_val), eval_tree(node[2], t_val))

def predict(node, t_vec):
    return np.array([eval_tree(node, ti) for ti in t_vec], dtype=float)

### TODO1: Fitness
COmplete the Fitness function defined as Fitness = MSE + λ * size

In [ ]:
# ============================================================
# TODO 1: Fitness = MSE + λ * size
# ============================================================
LAMBDA_SIZE = 3e-4

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

def fitness(node):
    """
    Compute the fitness of a tree.
    Steps:
    1. Predict values: y_hat = predict(node, t)
    2. Compute mean squared error (MSE) between y and y_hat.
    3. Add a small penalty for tree size (LAMBDA_SIZE * tree_size(node)).
    Return: fitness value.
    """
    # --- TODO: Fill in ---
    raise NotImplementedError()



### TODO 2: Selection (Tournament)

In [ ]:
# ============================================================
# TODO 2 Selection (Tournament)
# ============================================================
POP_SIZE     = 300
TOURN_SIZE   = 5
CROSS_RATE   = 0.9
MUT_RATE     = 0.25
GENERATIONS  = 120
ELITISM      = True


def tournament_select(pop, losses, k=TOURN_SIZE):
    """
    Select one individual by tournament:
    1. Randomly choose k individuals from population.
    2. Return the one with the lowest fitness (loss).
    """
    # --- TODO: Fill in ---
    raise NotImplementedError()

### TODO 3: Crossover (Subtree Exchange)

In [ ]:
# ============================================================
# Subtree utilities
# ============================================================
def all_subtrees(node):
    """Yield (parent, is_left, ref) for every subtree (root has parent=None)."""
    yield (None, None, node)
    if is_terminal(node): return
    op = node[0]; ar = FUN_SET[op][0]
    if ar == 1:
        for st in all_subtrees(node[1]): yield (node, True, st[2])
    else:
        for st in all_subtrees(node[1]): yield (node, True, st[2])
        for st in all_subtrees(node[2]): yield (node, False, st[2])

def replace_subtree(tree, target, repl):
    if tree is target: return repl
    if is_terminal(tree): return tree
    op = tree[0]; ar = FUN_SET[op][0]
    if ar == 1:
        child = replace_subtree(tree[1], target, repl)
        return (op, child)
    left = replace_subtree(tree[1], target, repl)
    right = replace_subtree(tree[2], target, repl)
    return (op, left, right)

## complete the crossover
def crossover(p1, p2):
    """
    Subtree crossover:
    1. With probability CROSS_RATE, pick a random subtree in each parent.
    2. Swap those subtrees to create two offspring.
    3. Otherwise, copy parents unchanged.
    """
    # --- TODO: Fill in ---
    raise NotImplementedError()


### TODO 4: Mutation

In [ ]:
# ============================================================
# TODO 4: Mutation
# ============================================================
def mutate(tree):
    """
    Mutation:
    1. With probability MUT_RATE, modify the tree.
    2. Option A: replace a random subtree with a new random one (grow()).
    3. Option B: slightly perturb a numeric constant leaf.
    """
    # --- TODO: Fill in ---
    raise NotImplementedError()


### Run GP

In [ ]:
# ============================================================
# Run GP
# ============================================================
population = [grow() for _ in range(POP_SIZE)]
best_hist = []

for gen in range(GENERATIONS):
    losses = [fitness(ind) for ind in population]
    bidx = int(np.argmin(losses))
    best = population[bidx]
    best_hist.append(losses[bidx])
    if gen % 5 == 0:
        print(f"Gen {gen:02d} | best loss = {losses[bidx]:.6f} | size = {tree_size(best)} | expr = {pretty(best)}")

    new_pop = [best] if ELITISM else []
    while len(new_pop) < POP_SIZE:
        p1 = tournament_select(population, losses)
        p2 = tournament_select(population, losses)
        c1, c2 = crossover(p1, p2)
        new_pop.extend([mutate(c1), mutate(c2)])
    population = new_pop[:POP_SIZE]

# ============================================================
# Plot Results
# ============================================================
final_losses = [fitness(ind) for ind in population]
best = population[int(np.argmin(final_losses))]
print("\nBest expression:", pretty(best))
print("Final loss:", fitness(best), "Size:", tree_size(best), "Depth:", tree_depth(best))

y_hat = predict(best, t)
plt.figure(figsize=(7,3))
plt.plot(t, y, 'k.', ms=2, alpha=0.5, label='data')
plt.plot(t, y_hat, 'r-', lw=2, label='GP best fit')
plt.legend(); plt.xlabel('t'); plt.ylabel('x(t)')
plt.title('Symbolic Regression — Damped Oscillator'); plt.tight_layout(); plt.show()

plt.figure(figsize=(5.3,3))
plt.plot(best_hist, lw=2)
plt.xlabel('Generation'); plt.ylabel('Best loss')
plt.title('GP Convergence'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()